# Promote, roll back and reproduce

Linden must separate a model artifact from the version currently used by the next batch job. This worked reference demonstrates aliases, validation tags and explicit application audit events.

**Local:** [set up once](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/docs/setup.md), select the module `.venv` kernel, then run cells in order. **Colab:** CPU, runtime version **2026.07** (Python 3.12). If setup asks for a restart after installing dependencies, use **Restart session**, then run again. No API key is needed. Setup prints a new temporary output folder; save your work before ending the session.

Use this one notebook throughout the sprint: predict → run the guided cells → add your answers in **Your turn** → finish the checkpoint. Campus notes explain the same work; you do not repeat it in a second notebook.

<details><summary>Campus notes for this sprint</summary>

- [From "which file is production?" to a registry](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-3/01-from-which-file-is-production-to-a-registry/lesson.md)
- [Registering and promoting a model](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-3/02-registering-and-promoting-a-model/lesson.md)
- [What full reproducibility requires](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-3/03-what-full-reproducibility-requires/lesson.md)
- [Reproducing a past training run](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-3/04-reproducing-a-past-training-run/lesson.md)
- [Rolling back a bad model version](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-3/05-rolling-back-a-bad-model-version/lesson.md)
- [Registry lifecycle deliverable](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-3/06-registry-lifecycle-deliverable/lesson.md)

</details>

In [1]:
from pathlib import Path
import json, tempfile, sys, subprocess
PIN = {'package': 'campus-mlops', 'version': '0.1.1', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'repository': 'https://github.com/richhiey/ml-llm-ops.git', 'source_commit': 'f1ae85914f49a4f1c4bdd6a0ca62c4c4076158ea', 'package_path': 'module-a'}
assert sys.version_info[:2] == (3, 12), 'Use Python 3.12; see the linked setup guide.'
if 'google.colab' in sys.modules:
    # Start before importing numerical packages: installed files cannot replace live modules.
    expected = {'numpy':'2.3.3', 'pandas':'2.3.3', 'sklearn':'1.7.2', 'mlflow':'3.5.1'}
    stale = [name for name, version in expected.items()
             if name in sys.modules and getattr(sys.modules[name], '__version__', None) != version]
    checkout = Path('/content') / ('ml-llm-ops-' + PIN['source_commit'][:12])
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--no-checkout', PIN['repository'], str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', PIN['source_commit']], check=True)
    actual = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'], text=True).strip()
    assert actual == PIN['source_commit']
    package = checkout / PIN['package_path']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(package/'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest==8.4.2'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(package)], check=True)
    if stale:
        raise RuntimeError('Pinned dependencies are now installed. Choose Runtime > Restart session, '
                           'then Run all again. Keep this runtime: do NOT disconnect/delete it. '
                           'Loaded packages needing refresh: ' + ', '.join(stale))
    # A running kernel does not reload newly installed editable .pth files.
    sys.path.insert(0, str(package / 'src'))
import campus_mlops
from campus_mlops import *
from campus_mlops.lifecycle import code_revision, MODEL_NAME
PACKAGE=Path(campus_mlops.__file__).resolve().parents[2]
sys.path.insert(0,str(PACKAGE))
from learning.workspace import make_project, project_python, project_tests, course_cli, prepare_case
assert campus_mlops.__version__ == PIN['version'] == '0.1.1'
assert code_revision() == PIN['code_revision'], 'Helper changed: restore the pinned reference source.'
state = Path(tempfile.mkdtemp(prefix='campus-module-a-'))
print('Helper', campus_mlops.__version__, 'source commit:', PIN['source_commit'], 'state:', state)


Helper 0.1.1 source commit: f1ae85914f49a4f1c4bdd6a0ca62c4c4076158ea state: /var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-b8xo8ggy


## Register a pending artifact

A run, logged model and registered version are different identities. Record each returned value. Registration must not silently become approval or active selection.

In [2]:
ws=Workspace(state/'tracking')
approved=ws.run(Config(alpha=1,range_guard=True),name='baseline-candidate')
v1=register(ws,approved['run_id'])
candidate=ws.client.get_model_version(MODEL_NAME,v1)
print(approved['run_id'],approved['model_uri'],v1,candidate.tags)
assert candidate.tags['validation_status']=='pending'
assert candidate.run_id==approved['run_id']

2026/10/19 17:38:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 17:38:19 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 17:38:19 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:19 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 17:38:19 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026/10/19 17:38:19 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 17:38:19 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:19 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2f5a4e71a7e7434baef53a09cc92b298 models:/m-69589ae468dd4fb5829cb0f0529c3b4d 1 {'validation_status': 'pending'}


Successfully registered model 'dev.linden-demand'.
Created version '1' of model 'dev.linden-demand'.


## Observe the pending boundary

Predict what happens if a pending version is selected. The helper applies a teaching policy; storing tags and aliases alone would not enforce this decision.

In [3]:
try:
    activate(ws,v1,'Deliberate pending-selection probe')
except ValueError as error:
    print('Expected rejection:',error)
else:
    raise AssertionError('Pending selection accepted')

Expected rejection: Only approved versions may be active


## Validate the observed behavior and select

Check a valid table is preserved and an invalid hour rejects. The approval helper also checks finite held-out RMSE and guard configuration; it does not execute your tests for you. Keep the observed rationale distinct from a magic approval label.

In [4]:
import pandas as pd
frame=ingest();pd.testing.assert_frame_equal(validate_ranges(frame),frame)
bad=frame.copy();bad.loc[0,'hour']=24
try:
    validate_ranges(bad)
except ValueError:
    print('Observed hour=24 rejection')
else:
    raise AssertionError('Invalid hour accepted')
approve(ws,v1,'Observed valid-copy/hour24 probes and finite validation RMSE<=5')
print(activate(ws,v1,'Validated baseline selected'))

Observed hour=24 rejection
{'timestamp_utc': '2026-10-19T15:38:19.703699+00:00', 'model': 'dev.linden-demand', 'alias': 'active', 'previous_version': None, 'new_version': '1', 'run_id': '2f5a4e71a7e7434baef53a09cc92b298', 'reason': 'Validated baseline selected'}


## Exercise a rollback scenario
A trial passes the original 5-kWh gate. A new acceptance brief tightens validation RMSE to 1.6 kWh; the alpha=10 trial fails it while the original alpha=1 passes. Record this reason and roll back the alias without deleting either model.

In [5]:
trial=ws.run(Config(alpha=10,range_guard=True),name='trial')
v2=register(ws,trial['run_id'])
approve(ws,v2,'Passed original 5-kWh teaching gate')
activate(ws,v2,'Trial before stricter acceptance review')
assert trial['metrics']['rmse_kwh']>1.6 and approved['metrics']['rmse_kwh']<1.6
ws.client.set_model_version_tag(MODEL_NAME,v2,'incident','Fails revised 1.6-kWh gate')
event=activate(ws,v1,'Rollback: trial fails revised 1.6-kWh acceptance')
print(event)
assert event['previous_version']==v2 and event['new_version']==v1

/Users/richhiey/Desktop/projects/masterschool/lessons/mlops-llmops/shared/ml-llm-ops/module-a/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/19 17:38:19 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a v

Registered model 'dev.linden-demand' already exists. Creating a new version of this model...
Created version '2' of model 'dev.linden-demand'.


{'timestamp_utc': '2026-10-19T15:38:19.878190+00:00', 'model': 'dev.linden-demand', 'alias': 'active', 'previous_version': '2', 'new_version': '1', 'run_id': '2f5a4e71a7e7434baef53a09cc92b298', 'reason': 'Rollback: trial fails revised 1.6-kWh acceptance'}


## Verify the next inference job
A rollback is only useful if inference uses it. The helper resolves the alias once and pins its version for this batch.

In [6]:
output,inference=batch_predict(ws)
print(inference)
print((ws.path/'audit.jsonl').read_text())
assert inference['resolved_version']==v1
assert ws.client.get_model_version(MODEL_NAME,v1).run_id==approved['run_id']

{'resolved_version': '1', 'run_id': '2f5a4e71a7e7434baef53a09cc92b298', 'model': 'dev.linden-demand', 'rows': 24, 'elapsed_seconds': 0.007891832967288792, 'rows_per_second': 3041.1185968429213, 'complete_within_60s': True, 'input_sha256': '5b74deb237ea9a085666ab0b3f865db8c7c996b935b38b24e829567d89142ee6'}
{"timestamp_utc": "2026-10-19T15:38:19.703699+00:00", "model": "dev.linden-demand", "alias": "active", "previous_version": null, "new_version": "1", "run_id": "2f5a4e71a7e7434baef53a09cc92b298", "reason": "Validated baseline selected"}
{"timestamp_utc": "2026-10-19T15:38:19.872520+00:00", "model": "dev.linden-demand", "alias": "active", "previous_version": "1", "new_version": "2", "run_id": "9b79caaad87347008ac06698bdcdb4bc", "reason": "Trial before stricter acceptance review"}
{"timestamp_utc": "2026-10-19T15:38:19.878190+00:00", "model": "dev.linden-demand", "alias": "active", "previous_version": "2", "new_version": "1", "run_id": "2f5a4e71a7e7434baef53a09cc92b298", "reason": "Rollb

## Reproduce the approved run
This checks source, data, config and environment before comparing RMSE. A new store is sufficient for a state replay; the clean-environment procedure additionally restores a source archive and a new venv.

The destination does not exist before this call. The original manifest identifies the recipe. `reproduce` checks current source/data/environment before fitting, then compares the same evaluation partition. New execution IDs are expected; a different snapshot is not. This reference is a new-store replay, not a new-environment proof.

In [7]:
report=reproduce(ws,approved['run_id'],state/'reproduced')
print(report)
assert report['passed'] and report['rmse_delta']<=1e-6
manifest=json.loads((ws.path/'evidence'/approved['run_id']/'manifest.json').read_text())
print({k:manifest[k] for k in ['run_id','code_revision','data_sha256','config','model_uri']})
print('Recorded installed distributions:',len(manifest['environment']['packages']))

2026/10/19 17:38:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 17:38:19 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 17:38:19 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:19 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 17:38:19 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026-10-19 17:38:20 INFO  [alembic.runtime.migration] Running upgrade 3da73c924c2f -> bf29a5ff90ea, add jobs table


2026-10-19 17:38:20 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:20 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


/Users/richhiey/Desktop/projects/masterschool/lessons/mlops-llmops/shared/ml-llm-ops/module-a/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/19 17:38:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a v

{'original_run_id': '2f5a4e71a7e7434baef53a09cc92b298', 'reproduced_run_id': '02c6cb1384b6475c895ef6cb7e9b6f18', 'rmse_delta': 0.0, 'tolerance': 1e-06, 'passed': True, 'lineage_matched': True}
{'run_id': '2f5a4e71a7e7434baef53a09cc92b298', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'data_sha256': '97fd0648a8277e2ae88f90bb20389745667df3df6b363beef23787ff427e5e54', 'config': {'alpha': 1, 'autolog': False, 'range_guard': True}, 'model_uri': 'models:/m-69589ae468dd4fb5829cb0f0529c3b4d'}
Recorded installed distributions: 119


## Inspect the source bundle before recovery

A hash cannot reconstruct missing bytes. Inspect the archive members and identify the actual source, data and lock needed for a fresh environment. Do not extract over your working project.

In [8]:
import tarfile
source=Path(ws.client.download_artifacts(approved['run_id'],'reproduction/source.tar.gz'))
with tarfile.open(source) as archive:
    members=archive.getnames()
print([name for name in members if name in ['pyproject.toml','uv.lock','src/campus_mlops/pipeline.py','src/campus_mlops/data/demand-v1.csv']])
assert all(name in members for name in ['pyproject.toml','uv.lock','src/campus_mlops/pipeline.py'])

['pyproject.toml', 'src/campus_mlops/data/demand-v1.csv', 'src/campus_mlops/pipeline.py', 'uv.lock']


## Your turn: recover a run and handle an incident

Use the run and manifests produced above for the past-run activity, retaining the original identity before replay. For independent rollback, prepare an incident with `prepare_case("rollback", state/"your-incident")`, then read its `case.json` and open `Workspace(state/"your-incident")`. This fixture leaves the trial active; you choose the response. Apply the revised 1.6-kWh client limit, inspect approval/source metrics, restore a suitable prior version and prove a new batch used it. Keep before/after selection and a reasoned audit note here. A new store replay proves state independence; fresh-environment recovery comes during handoff.

## Sprint checkpoint

Prepare the distinct Harbor incident with `prepare_case("registry-assessment", state/"harbor")`, read `case.json`, and inspect `Workspace(state/"harbor")`. The client limit is 1.45 kWh and the newest version is pending. Identify a qualifying prior approved version, restore it, tag the incident, run a new batch and reproduce its source run. Complete the Campus client brief without copying the worked rollback decision.

## Finish and keep your work

Save this notebook and the linked project/output files. Keep one sprint submission, reusing the work already completed. Temporary tracking directories disappear when the environment is discarded; save the files before closing it. Continue with the same project in the next sprint.